In [1]:
!pip install -q transformers accelerate bitsandbytes torch
!pip install -q langchain langchain-community langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 108.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.3/564.3 kB 36.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not

In [2]:
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 1.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.1/103.1 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.0/208.0 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 8.7 MB/s eta

In [ ]:
token = ""

In [4]:
from huggingface_hub import login

login(token)

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from langchain_community.llms import HuggingFacePipeline   # ✅ Sửa import ở đây
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
import torch
from time import time

time_start = time()


# --- 1. Tải Mô hình Llama 3.1 và Tokenizer ---
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

device = f'cuda:{torch.cuda.current_device()}' if torch.cuda.is_available() else 'cpu'

# Cấu hình lượng tử hóa 4-bit để tiết kiệm bộ nhớ
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map=device
)

print("🚀 Đang tải mô hình Llama 3.1 (có thể mất vài phút)...")
# 🔹 Tạo pipeline để wrap vào LangChain
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    top_p=0.9,
    repetition_penalty=1.1
)

# 🔹 Tạo LLM object cho LangChain
llm = HuggingFacePipeline(pipeline=pipe)

time_end = time()
print(f"Prepare model, tokenizer: {round(time_end-time_start, 3)} sec.")

2025-10-16 15:19:06.586503: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760627946.778187      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760627946.829083      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Device set to use cuda:0


🚀 Đang tải mô hình Llama 3.1 (có thể mất vài phút)...
Prepare model, tokenizer: 191.594 sec.


/tmp/ipykernel_19/970613940.py:44: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [6]:
import pandas as pd
from langchain_core.documents import Document

documents = []
# <-- Đặt đường dẫn file tri thức của bạn ở đây
knowledge_base_path = '/kaggle/input/ielts-dataset/train_final.csv'

try:
    df_knowledge = pd.read_csv(knowledge_base_path)
    print(f"Đã đọc thành công file tri thức. Tổng số ví dụ: {len(df_knowledge)}")

    # Chuyển đổi mỗi hàng trong DataFrame thành một đối tượng LangChain Document
    for index, row in df_knowledge.iterrows():
        # Kết hợp các phần văn bản quan trọng để tạo ngữ cảnh
        page_content = (
            f"Prompt: {row['prompt']}\n\n"
            f"Essay: {row['essay']}\n\n"
            f"Scores:\n"
            f"  - Task Response (TR): {row['TR_Band']}\n"
            f"  - Coherence & Cohesion (CC): {row['CC_Band']}\n"
            f"  - Lexical Resource (LR): {row['LR_Band']}\n"
            f"  - Grammatical Range & Accuracy (GRA): {row['GRA_Band']}"
        )
        
        # Metadata bây giờ chỉ chứa thông tin về điểm số (band)
        metadata = {
            "TR_Band": row['TR_Band'],
            "CC_Band": row['CC_Band'],
            "LR_Band": row['LR_Band'],
            "GRA_Band": row['GRA_Band']
        }
        documents.append(Document(page_content=page_content, metadata=metadata))

    print(f"Đã tạo {len(documents)} documents từ dữ liệu tri thức.")

except FileNotFoundError:
    print(f"LỖI: Không tìm thấy file tri thức tại '{knowledge_base_path}'. Vui lòng kiểm tra lại đường dẫn.")
    documents = [] # Đảm bảo documents là list rỗng để không bị lỗi ở các bước sau
except Exception as e:
    print(f"Đã xảy ra lỗi khi đọc file tri thức: {e}")
    documents = []

Đã đọc thành công file tri thức. Tổng số ví dụ: 9833
Đã tạo 9833 documents từ dữ liệu tri thức.


In [7]:
documents[:2]

[Document(metadata={'TR_Band': 8.0, 'CC_Band': 8.0, 'LR_Band': 7.0, 'GRA_Band': 7.0}, page_content='Prompt: Interviews form the basic criteria for most large companies. However, some people think that the interview is not a reliable method of choosing whom to employ and there are other better methods. To what extent do you agree or disagree?\n\nEssay: It is believed by some experts that the traditional approach of recruiting candidates which is interviewing is the best way, whereas others think different methods such as exams writing, CVs, cover letters or application letters and many more are good. I strongly agree with the statement, "interview is the most reliable approach to recruit workers" because this method assists the recruiters to know the person and his ability to do the work and their problem-solving abilities. \n\nTo begin with, an interview enables the  recruiter to know the kind of person he or she is recruiting. It helps the employer to see the personality traits of the

In [8]:
from langchain_community.embeddings import HuggingFaceEmbeddings


model_name = "sentence-transformers/all-mpnet-base-v2"
model_kwargs = {"device": "cuda"}

# try to access the sentence transformers from HuggingFace: https://huggingface.co/api/models/sentence-transformers/all-mpnet-base-v2
try:
    embeddings = HuggingFaceEmbeddings(model_name=model_name, model_kwargs=model_kwargs)
except Exception as ex:
    print("Exception: ", ex)
    # alternatively, we will access the embeddings models locally
    local_model_path = "/kaggle/input/sentence-transformers/minilm-l6-v2/all-MiniLM-L6-v2"
    print(f"Use alternative (local) model: {local_model_path}\n")
    embeddings = HuggingFaceEmbeddings(model_name=local_model_path, model_kwargs=model_kwargs)

/tmp/ipykernel_19/1551073730.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=model_name, model_kwargs=model_kwargs)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
from langchain_community.vectorstores import Chroma
vectordb = Chroma.from_documents(documents=documents, embedding=embeddings, persist_directory="chroma_db")

In [10]:
qa_chain = None
if documents:
    prompt_template = """
    You are a highly experienced IELTS writing examiner. Your goal is to provide a precise and consistent evaluation of an essay by following a structured reasoning process.
    
    **CONTEXT (Reference Essays with Scores):**
    {context}
    
    **NEW ESSAY TO GRADE:**
    {question}

    **EVALUATION PROCESS (Think step-by-step):**
    
    1.  **Task Response (TR) Analysis:**
        * Briefly assess how well the 'NEW ESSAY' addresses the prompt.
        * Compare its quality to the Task Response (TR) scores in the 'CONTEXT'.
        * Conclude with a final Task Response (TR) band score.
    
    2.  **Coherence and Cohesion (CC) Analysis:**
        * Briefly assess the essay's structure, paragraphing, and use of linking devices.
        * Compare its quality to the Coherence & Cohesion (CC) scores in the 'CONTEXT'.
        * Conclude with a final Coherence & Cohesion (CC) band score.
    
    3.  **Lexical Resource (LR) Analysis:**
        * Briefly assess the range and accuracy of vocabulary used.
        * Compare its quality to the Lexical Resource (LR) scores in the 'CONTEXT'.
        * Conclude with a final Lexical Resource (LR) band score.
    
    4.  **Grammatical Range and Accuracy (GRA) Analysis:**
        * Briefly assess the range and accuracy of grammatical structures.
        * Compare its quality to the Grammatical Range & Accuracy (GRA) scores in the 'CONTEXT'.
        * Conclude with a final Grammatical Range & Accuracy (GRA) band score.

    FINAL OUTPUT:
    Provide ONLY ONE JSON object — nothing else. 
    
    **CRITICAL RULES:**
    - Do NOT include explanations, Markdown, or repeated JSON.
    - Your entire output MUST be a single valid JSON object with these keys:
      "TR_Band", "CC_Band", "LR_Band", "GRA_Band".
    - Each value must be a float between 0.0 and 9.0, in 0.5 increments.
    
    JSON Response:
    """
    PROMPT = PromptTemplate(template=prompt_template, input_variables=["context", "question"])

    retriever = vectordb.as_retriever(search_kwargs={'k': 2}) # Lấy 2 ví dụ liên quan nhất

    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        return_source_documents=False,
        chain_type_kwargs={"prompt": PROMPT},
        verbose=False
    )
    print("RAG chain đã sẵn sàng.")
else:
    print("Không thể tạo RAG chain vì không có dữ liệu tri thức.")

RAG chain đã sẵn sàng.


In [11]:
import json
import time
import re # <-- Thêm thư viện này vào

def grade_ielts_essay(chain, prompt, essay, row):
    """
    Chạy inference để chấm điểm và trả về kết quả dạng dictionary.
    Hàm này sử dụng Regex để trích xuất JSON một cách mạnh mẽ hơn.
    """
    if not chain:
        return {"error": "RAG chain is not available."}

    query = f"Prompt: {prompt}\n\nEssay: {essay}"
    
    start_time = time.time()
    result = chain({"query": query})
    end_time = time.time()
    
    response_time = round(end_time - start_time, 2)
    
    try:
        # ---- BẮT ĐẦU PHẦN THAY ĐỔI ----
        # Sử dụng Regex để tìm chuỗi JSON nằm giữa { và }
        # re.DOTALL cho phép '.' khớp với cả ký tự xuống dòng
        rag_response_str = str(result.get("result", ""))  # hoặc "answer" nếu bạn dùng RetrievalQA dạng khác

        pattern = r'JSON Response:\s*(\{\s*"TR_Band"\s*:\s*[\d.]+,\s*"CC_Band"\s*:\s*[\d.]+,\s*"LR_Band"\s*:\s*[\d.]+,\s*"GRA_Band"\s*:\s*[\d.]+\s*\})'
        
        match = re.search(pattern, rag_response_str)
        
        if match:
            json_str = match.group(1)
            parsed_result = json.loads(json_str)
            print(parsed_result)
        else:
            print("Không tìm thấy JSON hợp lệ sau 'JSON Response:'.")
        

        bands = [
            parsed_result.get("TR_Band", 0), 
            parsed_result.get("CC_Band", 0),
            parsed_result.get("LR_Band", 0), 
            parsed_result.get("GRA_Band", 0)
        ]

        parsed_result['actual_brand'] = row["Overall_Band"]
        
        if all(isinstance(b, (int, float)) for b in bands):
            avg_band = sum(bands) / 4.0
            final_band = round(avg_band * 2) / 2
            parsed_result['predicted_brand'] = final_band
        else:
            parsed_result['predicted_brand'] = "Error: Invalid band scores"
            
        return parsed_result
        
    except (json.JSONDecodeError, IndexError, ValueError) as e:
        return {
            "error": f"Failed to parse LLM output: {e}",
            "raw_response": result['result'],
        }

In [12]:
from IPython.display import display, Markdown

# %% [markdown]
# ---
# ## Phần 5: Chạy Inference trên toàn bộ file test

# %% [code]
test_file_path = '/kaggle/input/ielts-dataset/test_final.csv'
all_results = []

try:
    df_test = pd.read_csv(test_file_path)
    print(f"Đã đọc thành công file test. Tổng số bài luận cần chấm: {len(df_test)}")
    
    for index, row in df_test.iterrows():
        prompt = row['prompt']
        essay = row['essay']
        
        display(Markdown(f"### 📝 Đang chấm bài luận #{index + 1}..."))
        display(Markdown(f"**Prompt:** {prompt[:100]}..."))
        
        grading_result = grade_ielts_essay(qa_chain, prompt, essay, row)
        
        display(Markdown("#### ✅ Kết quả chấm điểm:"))
        display(Markdown(f"```json\n{json.dumps(grading_result, indent=2)}\n```"))
        
        all_results.append(grading_result)
        
        display(Markdown("---"))

except FileNotFoundError:
    print(f"Lỗi: Không tìm thấy file test tại đường dẫn '{test_file_path}'")
except Exception as e:
    print(f"Đã xảy ra lỗi không mong muốn: {e}")

Đã đọc thành công file test. Tổng số bài luận cần chấm: 495


### 📝 Đang chấm bài luận #1...

**Prompt:** Some people think that the best way to solve global environmental problems is to increase the cost o...

/tmp/ipykernel_19/911458881.py:16: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = chain({"query": query})
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


{'TR_Band': 8.5, 'CC_Band': 8.5, 'LR_Band': 7.5, 'GRA_Band': 7.5}


...

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


{'TR_Band': 7.0, 'CC_Band': 7.0, 'LR_Band': 6.5, 'GRA_Band': 7.0}


#### ✅ Kết quả chấm điểm:

```json
{
  "TR_Band": 7.0,
  "CC_Band": 7.0,
  "LR_Band": 6.5,
  "GRA_Band": 7.0,
  "actual_brand": 6.0,
  "predicted_brand": 7.0
}
```

---

In [13]:
# %% [markdown]
# ---
# ## Phần 6: Lưu kết quả

# %% [code]
if all_results:
    output_json_path = 'grading_results.json'
    with open(output_json_path, 'w') as f:
        json.dump(all_results, f, indent=2)
    print(f"Đã lưu thành công {len(all_results)} kết quả vào file '{output_json_path}'")

    output_csv_path = 'grading_results.csv'
    df_results = pd.DataFrame(all_results)
    df_results.to_csv(output_csv_path, index=False)
    print(f"Đã lưu thành công {len(all_results)} kết quả vào file '{output_csv_path}'")

Đã lưu thành công 495 kết quả vào file 'grading_results.json'
Đã lưu thành công 495 kết quả vào file 'grading_results.csv'
